# Flight Emissions Data

We get flight data from OpenSky. We merge that with Europa data about emissions intensity by aircraft model.


## Imports and Constants

In [1]:
import datetime as dt
from datetime import date, time, datetime
from zoneinfo import ZoneInfo
import json
from time import sleep
# import json
# from pprint import pprint
# import urllib.request 
import os
# from statistics import median, mean
import logging

from requests import HTTPError, ReadTimeout
from tqdm import tqdm
import polars as pl
from opensky_api import OpenSkyApi
import numpy as np

In [97]:

# will be created if doesn't exist
data_dir = "data"

# from here: https://opensky-network.org/datasets/#metadata/aircraftDatabase
mapping_csv = os.path.join(data_dir, "raw/aircraftDatabase.csv.gz")

# from here: https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/view
# https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/at_download/file
emissions_spreadsheet = os.path.join(data_dir, "raw/1.A.3.a Aviation -Annex 1 - Master emissions calculator - 2023 - Protected - v1.5_18_09_2024.xlsx")

# downloaded from
# https://data.cso.ie/
# discovered from Wikipedia: https://en.wikipedia.org/wiki/Dublin_Airport#Statistics
passenger_numbers_sheet = os.path.join(data_dir, "popular_destinations.csv")

opensky_api_secret_file = "/home/matthew/.local/share/credentials/opensky_api_key.json"

results_dir = os.path.join(data_dir, "results")
emissions_results_path = os.path.join(results_dir, "emissions.parquet")
planes_results_path = os.path.join(results_dir, "planes.parquet")
passenger_numbers_results_path = os.path.join(results_dir, "passengers-per-flight.txt")

In [3]:
# microseconds per second
us_per_s = 1e6

# Melbourne and Sydney are the same timezone
tz_name = 'Australia/Sydney'


h_per_day = 24

In [4]:
with open("airports.json", "r") as f:
    airport_info = json.load(f)

airport_id_type = "ICAO"
airport_ids = [v[airport_id_type] for (k, v) in airport_info.items()]

In [5]:
# Canberra and Melbourne are in the same time zone
sydney_tz = ZoneInfo(tz_name)

In [59]:
taxiing_time = dt.timedelta(minutes=15) # arbitrary

## Download flight data from Open Sky

The OpenSky API is documented [here](https://openskynetwork.github.io/opensky-api/python.html#opensky_api.FlightData).


In [6]:
# logger = logging.getLogger("opensky_api")
# #logger.addHandler(logging.StreamHandler())
# logger.setLevel(logging.DEBUG)

logger = logging.getLogger("opensky_api")
logger.setLevel(logging.ERROR)
handler = logging.StreamHandler()
logger.addHandler(handler)

In [7]:
api = OpenSkyApi(client_json_path=opensky_api_secret_file)

In [8]:
def get_epoch(t: dt.datetime) -> int:
    return round(t.timestamp())

In [9]:
today = dt.date.today()


# OpenSky does some really weird stuff with time zones and day bounds, so select several days of data
# and then take the 3rd Sydney calendar day.
_dates = [today - dt.timedelta(days=d) for d in range(5, 5+4)]
_date = _dates[2]
(_dates, _date)

([datetime.date(2026, 2, 23),
  datetime.date(2026, 2, 22),
  datetime.date(2026, 2, 21),
  datetime.date(2026, 2, 20),
  datetime.date(2026, 2, 19),
  datetime.date(2026, 2, 18)],
 datetime.date(2026, 2, 21))

In [10]:


def get_response(airport, begin, end, tries=10):
    try:
        return api.get_departures_by_airport(
            airport=airport, 
            begin=get_epoch(begin), 
            end=get_epoch(end)
        )
    except (TimeoutError, ReadTimeout):
        if tries > 0:
            print(f"Timeout {airport=} {begin=} to {end=}")
            sleep(60)
            return get_response(airport, begin, end, tries=tries-1)
        else:
            print(f"Issue with {airport=} {begin=} to {end=}")
            raise
    except HTTPError as e:
        print(f"Issue with {airport=} {begin=} to {end=}: {e}")
        if tries > 0:
            sleep(10)
            return get_response(airport, begin, end, tries=tries-1)
        else:
            print(f"Issue with {airport=} {begin=} to {end=}")
            raise
        


In [17]:
responses = []

from itertools import pairwise
# query in overlapping windows of 2 UTC days
for left_date, right_date in pairwise(sorted(_dates)):
    dt_start = dt.datetime.combine(left_date, dt.time.min, tzinfo=dt.timezone.utc) # time(hour=1)
    dt_end = dt.datetime.combine(right_date, dt.time.max, tzinfo=dt.timezone.utc)


    for airport in tqdm(airport_ids):
        response = get_response(airport, dt_start, dt_end)
        responses.extend(response)

  0%|                                                                                                                                                             | 0/4 [00:00<?, ?it/s]

Timeout airport='YMML' begin=datetime.datetime(2026, 2, 18, 0, 0, tzinfo=datetime.timezone.utc) to end=datetime.datetime(2026, 2, 19, 23, 59, 59, 999999, tzinfo=datetime.timezone.utc)


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:15<00:00,  3.95s/it]


In [ ]:
# responses = []

# window_width = dt.timedelta(hours=12)
# window_shift = dt.timedelta(hours=6)

# dt_start = dt.datetime.combine(min(_dates), dt.time.min)
# dt_end = dt.datetime.combine(max(_dates), dt.time.max)
# t = dt_start

# windows = []


# while t < dt_end:
#     windows.append((t, t + window_width))    
#     t += window_shift


# for airport in airport_ids:
#     for (t_start, t_end) in tqdm(windows):
    
#         response = get_response(airport, t_start, t_end)
#         responses.extend(response)
#         # print(f"Got response for {airport} {t}")


In [18]:
flight_api_data = pl.LazyFrame([f.__dict__ for f in responses])
flight_api_data.sink_csv(os.path.join(data_dir, "api_raw.csv"))
flight_api_data.sink_parquet(os.path.join(data_dir, "api_raw.parquet"))

## Process Flight Data

You can skip the previous section and start from here to read cached data from disk.

Parse datetimes, calculate duration. (e.g. `firstSeen` actually is departure time.)
We delete in-progress flights, because that duration will be wrong.

In [19]:
flights = (
    pl.scan_parquet(os.path.join(data_dir, "api_raw.parquet"))
    .filter(pl.col("estArrivalAirport").is_not_null())
    .with_columns(
        (pl.col("firstSeen") * us_per_s).cast(
            pl.Datetime()
        ).dt.convert_time_zone("Australia/Sydney"),
        (pl.col("lastSeen") * us_per_s).cast(
            pl.Datetime()
        ).dt.convert_time_zone("Australia/Sydney"),
    )
    .with_columns(
        (pl.col("lastSeen") - pl.col("firstSeen")).alias("flightDurationActual")
    )
    .select(
        pl.col("icao24").alias("ICAO24_HEX"),
        pl.col("firstSeen").alias("DEPARTURE_TIME"),
        pl.col("lastSeen").alias("ARRIVAL_TIME"),
        pl.col("flightDurationActual").alias("FLIGHT_DURATION_ACTUAL"),
        pl.col("estDepartureAirport").alias("DEPARTURE_AIRPORT"),
        pl.col("estArrivalAirport").alias("ARRIVAL_AIRPORT"),
    )
    .unique() # duplicates exist because of overlapping windows earlier
)
flights.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str
"""7c788b""",2026-02-19 10:12:37 AEDT,2026-02-19 11:30:23 AEDT,1h 17m 46s,"""YSSY""","""YMML"""
"""7c6b35""",2026-02-22 06:46:33 AEDT,2026-02-22 12:49:28 AEDT,6h 2m 55s,"""YMML""","""YMML"""
"""750523""",2026-02-22 22:30:12 AEDT,2026-02-23 06:24:33 AEDT,7h 54m 21s,"""YSSY""","""WMKK"""
"""7c56b7""",2026-02-23 22:04:45 AEDT,2026-02-23 22:45:58 AEDT,41m 13s,"""YMML""","""YSCB"""
"""c82759""",2026-02-22 19:39:13 AEDT,2026-02-22 22:33:19 AEDT,2h 54m 6s,"""YMML""","""NZCH"""


In [20]:
# check we've got 10am-12am, because of timezone/midnight issue
(
    flights
    .with_columns(
        pl.col("ARRIVAL_TIME").dt.hour().alias("HOUR"),
        pl.col("ARRIVAL_TIME").dt.date().alias("DATE")
    )
    .filter(pl.col("HOUR").is_between(8, 13))
    .group_by("DATE", "HOUR")
    .len("COUNT")
    .sort("DATE")
    .collect()
    .pivot(
        "DATE",
        values="COUNT",   # what will fill the cells
        index="HOUR",     # one row per hour
    )
    .sort("HOUR")
)

HOUR,2026-02-18,2026-02-19,2026-02-20,2026-02-21,2026-02-22,2026-02-23,2026-02-24
i8,u32,u32,u32,u32,u32,u32,u32
8,null,36,42,37,26,46,43
9,null,51,45,41,29,44,48
10,null,40,40,40,39,52,41
11,4,54,50,50,42,49,44
12,15,52,48,41,43,36,20
13,34,39,35,40,36,39,6


In [21]:
# find a date which doesn't have a gap
_date = (
    flights
    .with_columns(
        pl.col("ARRIVAL_TIME").dt.hour().alias("HOUR"),
        pl.col("ARRIVAL_TIME").dt.date().alias("DATE")
    )
    .filter(pl.col("DATE") < pl.col("DATE").max())
    .filter(pl.col("DATE") > pl.col("DATE").min())
    .group_by("DATE", "HOUR")
    .len("COUNT")
    .group_by("DATE")
    .agg(pl.col("COUNT").is_not_null().all().alias("COMPLETE"))
    .select(pl.col("DATE").min())
    .collect()
    .item()
)
_date

datetime.date(2026, 2, 19)

In [22]:
flights = flights.filter(pl.col("ARRIVAL_TIME").dt.date() == _date)

In [23]:
(
    flights
    .select(
        pl.col("ARRIVAL_TIME").min().alias("ARRIVAL_MIN"),
        pl.col("ARRIVAL_TIME").max().alias("ARRIVAL_MAX"),
        pl.col("DEPARTURE_TIME").min().alias("DEPARTURE_MIN"),
        pl.col("DEPARTURE_TIME").max().alias("DEPARTURE_MAX"),
    )
    .collect()
)

ARRIVAL_MIN,ARRIVAL_MAX,DEPARTURE_MIN,DEPARTURE_MAX
"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]"
2026-02-19 00:05:15 AEDT,2026-02-19 23:57:52 AEDT,2026-02-18 11:15:38 AEDT,2026-02-19 22:42:10 AEDT


In [24]:
# filter to flights between Melbourne, Sydney, Canberra
# (raw data is departures from these, to anywhere. Filter to these too.)
# and during the time window (API might return extra)
flights = (
    flights
    .filter(
        pl.all_horizontal(
            pl.col("DEPARTURE_AIRPORT").is_in(airport_ids),
            pl.col("ARRIVAL_AIRPORT").is_in(airport_ids),
            pl.col("DEPARTURE_AIRPORT") != pl.col("ARRIVAL_AIRPORT"),
        )
    )
    # .filter(pl.col("DEPARTURE_TIME") >= time_start)
    # .filter(pl.col("ARRIVAL_TIME") >= time_end)
)
flights.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str
"""7c6b07""",2026-02-19 12:20:33 AEDT,2026-02-19 13:08:39 AEDT,48m 6s,"""YSCB""","""YMML"""
"""7c6d9b""",2026-02-19 16:11:01 AEDT,2026-02-19 17:36:59 AEDT,1h 25m 58s,"""YSSY""","""YMML"""
"""7c47b8""",2026-02-19 06:40:03 AEDT,2026-02-19 07:50:16 AEDT,1h 10m 13s,"""YSSY""","""YMML"""
"""7c6db4""",2026-02-19 17:53:33 AEDT,2026-02-19 19:13:08 AEDT,1h 19m 35s,"""YSSY""","""YMML"""
"""7c7c98""",2026-02-19 11:00:57 AEDT,2026-02-19 12:22:24 AEDT,1h 21m 27s,"""YSSY""","""YMML"""


In [25]:
flights.select("DEPARTURE_AIRPORT", "ARRIVAL_AIRPORT").unique().head().collect()

DEPARTURE_AIRPORT,ARRIVAL_AIRPORT
str,str
"""YSCB""","""YSSY"""
"""YSCB""","""YMAV"""
"""YSCB""","""YMML"""
"""YSSY""","""YSCB"""
"""YMML""","""YSCB"""


In [26]:
#What's the start time each day?
(
    flights
    .filter(pl.col("DEPARTURE_TIME").dt.date() > pl.col("DEPARTURE_TIME").dt.date().min())
    .group_by("DEPARTURE_AIRPORT")
    .agg(pl.col("DEPARTURE_TIME").min())
    .sort("DEPARTURE_TIME")
    .collect()
)

DEPARTURE_AIRPORT,DEPARTURE_TIME
str,"datetime[μs, Australia/Sydney]"
"""YSSY""",2026-02-19 00:32:30 AEDT
"""YMML""",2026-02-19 02:29:30 AEDT
"""YMAV""",2026-02-19 06:01:32 AEDT
"""YSCB""",2026-02-19 06:16:50 AEDT


## Join to emissions data

We download a mapping file from OpenSky, to map icao24 IDs to flight model.
Browse the files [here](https://opensky-network.org/datasets/#metadata/aircraftDatabase).
With that page, `aircraftDatabase.csv` came from [here]("https://s3.opensky-network.org/data-samples/metadata/aircraftDatabase.csv").

Within this CSV:

* `icao24` matches the OpenSki API
* `typecode` matches what's in the emissions spreadsheet as `ICAO_24`

In [27]:
def download_file(url, path):
    dir_path = os.path.dirname(path)
    os.makedirs(dir_path, exist_ok=True)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)

In [28]:
# https://opensky-network.org/datasets/#metadata/aircraftDatabase
url = "https://s3.opensky-network.org/data-samples/metadata/aircraftDatabase.csv"
download_file(url, mapping_csv)

In [29]:
mapping_df = (
    pl.scan_csv(mapping_csv, skip_rows_after_header=1)
    .select(
        pl.col("icao24").alias("ICAO24_HEX"),
        pl.col("typecode").alias("ICAO_OTHER")
    )
)
mapping_df.head().collect()

ICAO24_HEX,ICAO_OTHER
str,str
"""aa3487""","""BE36"""
"""a4fa61""","""PA31"""
"""a7a809""","""AC90"""
"""391927""","""DR40"""
"""503c21""","""ZZZZ"""


In [30]:
flight_and_type = flights.join(mapping_df, on="ICAO24_HEX", how="left")
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str
"""7c6c53""",2026-02-19 14:25:48 AEDT,2026-02-19 15:31:49 AEDT,1h 6m 1s,"""YMML""","""YSSY""","""B738"""
"""7c7a3e""",2026-02-19 08:02:17 AEDT,2026-02-19 09:06:52 AEDT,1h 4m 35s,"""YMML""","""YSSY""","""B738"""
"""7c7ab2""",2026-02-19 15:56:40 AEDT,2026-02-19 17:20:21 AEDT,1h 23m 41s,"""YSSY""","""YMML""","""B738"""
"""7c6d8d""",2026-02-19 08:00:02 AEDT,2026-02-19 09:05:18 AEDT,1h 5m 16s,"""YSSY""","""YMML""","""B738"""
"""06a078""",2026-02-19 12:23:39 AEDT,2026-02-19 13:14:34 AEDT,50m 55s,"""YSCB""","""YMML""","""B77W"""


In [31]:
# from here: https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/view
url = "https://www.eea.europa.eu/publications/emep-eea-guidebook-2023/part-b-sectoral-guidance-chapters/1-energy/1-a-combustion/1-a-3-a-aviation.3/at_download/file"
download_file(url, emissions_spreadsheet)

In [32]:
# This sheet in the file is hidden in Excel
# Extract it manually.
(
    pl.read_excel(
        source=emissions_spreadsheet,
        sheet_name="database",
        read_options={
            "skip_rows": 1,
            "header_row": True,
        },
        schema_overrides={"FORECAST  DURATION": pl.Duration()},
    )
    .write_excel(os.path.join(data_dir, "spreadsheet-extracted.xlsx"))
)

In [33]:
# load the emissions spreadsheet
emissions_lookup = (
    pl.read_excel(
        source=emissions_spreadsheet,
        sheet_name="database",
        read_options={
            "skip_rows": 1,
            "header_row": True,
        },
        schema_overrides={"FORECAST  DURATION": pl.Duration()},
    )
    .lazy()
    .filter(pl.col("ICAO_ID").is_not_null())
    .filter(pl.col("ICAO_ID") != "")
    .rename(
        {
            "ICAO_ID": "ICAO_OTHER",
            "AIRCRAFT ID": "AIRCRAFT_ID",
            # watch out, there are double spaces and trailing spaces
            # in the raw data
            "FORECAST  DURATION": "DURATION_REFERENCE",
            "FORECAST CO2 (3,15 for JET-A or 3,10 for AvGAS) ": "CO2",
            "FORECAST  NOX": "NOX",
            "FORECAST  SOX": "SOX",
            "FORECAST  H20": "H2O",
            "FORECAST  CO": "CO",
            "FORECAST  HC": "HC",
            " PM Non Volatile": "PM_NON_VOLATILE",
            "PM VOLATILE (all)": "PM_VOLATILE",
            "PM TOTAL": "PM_TOTAL",
        }
    )
)

emissions_cols = [
    "CO2",
    "NOX",
    "SOX",
    "H2O",
    "CO",
    "HC",
    "PM_NON_VOLATILE",
    "PM_VOLATILE",
    "PM_TOTAL",
]

emissions_lookup.head().collect()

ICAO_OTHER,AIRCRAFT_ID,IMPACT ACFT ID,Manufacturer,One of the models associated with this aircraft type,Description,Engine Type,Number of engines,ENGINE ID LTO,LTO or CCD,CRUISE ALT,ADES,DURATION_REFERENCE,FORECAST FUEL BURNT KG,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL
str,str,str,str,str,str,str,str,str,str,i64,i64,duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""A124""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",240,200,34m 1s,6430.264058,20255.331782,168.87469,5.401423,7954.237657,8.272752,3.778494,0.22115,0.689774,0.910925
"""A140""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",300,250,40m 42s,7665.318022,24145.75177,202.240391,6.438867,9481.998763,10.541993,4.727702,0.22849,0.873995,1.102485
"""A148""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",320,500,1h 16m 56s,14398.156664,45354.193491,338.863846,12.094451,17810.519114,16.794008,8.565683,0.350165,1.822895,2.17306
"""A19N""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",340,750,1h 53m 50s,21058.965284,66335.740645,475.476091,17.689524,26049.935059,23.35394,12.527249,0.43895,2.823369,3.262319
"""A20N""","""A124-A""","""A124""","""ANTONOV""","""AN-124 RUSLAN""","""LANDPLANE""","""JET""","""4""","""1RR005""","""CCD""",340,1000,2h 30m 19s,27751.271402,87416.504915,609.057052,23.311065,34328.321892,29.31615,16.318223,0.548512,3.792441,4.340953


Now we do the joins.

Some ICAO_OTHER hex values have a mapping to CCD but not LTO. Use that to get the Aircraft ID. Then look up aircraft ID if there's no match based on ICAO_OTHER.

"LTO" is landing and take off (the emissions while at the airport). "CCD" is cruise, control and descent (mid-flight).
I don't know what `LTO2` is, which is sometimes missing. So we just use `LTO`.

In [34]:
# look up takeoff and landing
flight_and_type = (
    flight_and_type
    .join(
        emissions_lookup
        .select("ICAO_OTHER", "AIRCRAFT_ID")
        .unique()
        , on="ICAO_OTHER", how="left"
    )
)
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str
"""7cb065""",2026-02-19 11:06:57 AEDT,2026-02-19 11:46:23 AEDT,39m 26s,"""YSCB""","""YSSY""",null,null
"""7c7886""",2026-02-19 12:39:58 AEDT,2026-02-19 14:00:59 AEDT,1h 21m 1s,"""YSSY""","""YMAV""",null,null
"""7cad4a""",2026-02-19 13:18:06 AEDT,2026-02-19 14:26:27 AEDT,1h 8m 21s,"""YMML""","""YSSY""",null,null
"""7c6dd5""",2026-02-19 20:35:38 AEDT,2026-02-19 21:37:24 AEDT,1h 1m 46s,"""YMML""","""YSSY""","""B738""","""A21N-A"""
"""7c0460""",2026-02-19 10:10:57 AEDT,2026-02-19 11:27:28 AEDT,1h 16m 31s,"""YSSY""","""YMML""",null,null


In [35]:
# look up takeoff and landing emissions
flight_and_type = (
    flight_and_type
    .join(
        emissions_lookup
        .filter(pl.col("LTO or CCD") == "LTO")
        .select(["AIRCRAFT_ID"] + [pl.col(c).alias(c + "_LTO") for c in emissions_cols])
        , on="AIRCRAFT_ID", how="left"
    )
)
flight_and_type.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7c68a5""",2026-02-19 12:06:02 AEDT,2026-02-19 18:14:25 AEDT,6h 8m 23s,"""YMML""","""YSSY""","""E190""","""A321-A""",3385.935,16.751387,0.902916,1329.6514,5.110164,0.073799,0.188192,0.054218,0.24241
"""7c7a36""",2026-02-19 18:28:05 AEDT,2026-02-19 19:51:50 AEDT,1h 23m 45s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992
"""7c6db8""",2026-02-19 09:11:31 AEDT,2026-02-19 10:27:30 AEDT,1h 15m 59s,"""YSSY""","""YMML""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992
"""7c6c53""",2026-02-19 14:25:48 AEDT,2026-02-19 15:31:49 AEDT,1h 6m 1s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992
"""7c56b7""",2026-02-19 17:16:24 AEDT,2026-02-19 17:57:41 AEDT,41m 17s,"""YMML""","""YSCB""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992


Cruise, control and descent (CCD) is harder. Most have many matches, for varying duration/distance. We need to match to two rows, and linearly interpolate in between them.


In [36]:
emissions_options = (
    flight_and_type
    .join(emissions_lookup,
          on="AIRCRAFT_ID",
          how="left"
    )
)

In [37]:
emissions_lower = (
    emissions_options
    .sort("DURATION_REFERENCE", descending=False)
    .filter(pl.col("DURATION_REFERENCE") >= pl.col("FLIGHT_DURATION_ACTUAL"))
    .group_by("AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL")
    .first()
    .select(
        ["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL", pl.col("DURATION_REFERENCE").alias("DURATION_REFERENCE_LOWER")]
        + [pl.col(c).alias(c + "_LOWER") for c in emissions_cols]
    )
)

emissions_upper = (
    emissions_options
    .sort("DURATION_REFERENCE", descending=True)
    .filter(pl.col("DURATION_REFERENCE") <= pl.col("FLIGHT_DURATION_ACTUAL"))
    .group_by("AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL")
    .first()
    .select(
        ["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL", pl.col("DURATION_REFERENCE").alias("DURATION_REFERENCE_UPPER")]
        + [pl.col(c).alias(c + "_UPPER") for c in emissions_cols]
    )
)

In [38]:
flight_emissions = (
    flight_and_type
    .join(emissions_lower, on=["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL"], how="left")
    .join(emissions_upper, on=["AIRCRAFT_ID", "FLIGHT_DURATION_ACTUAL"], how="left")
    .with_columns(
        ((pl.col("FLIGHT_DURATION_ACTUAL") - pl.col("DURATION_REFERENCE_LOWER")) / (pl.col("DURATION_REFERENCE_UPPER") - pl.col("DURATION_REFERENCE_LOWER"))).alias("INTERPOLATION_FACTOR")
    )
    .with_columns(
        [
            (pl.col(c + "_LOWER") + pl.col("INTERPOLATION_FACTOR") * (pl.col(c + "_UPPER") - pl.col(c + "_LOWER"))).alias(c + "_CCD")
            for c in emissions_cols
        ]
    )
    .select(
        pl.exclude([c + app for c in emissions_cols for app in ["_UPPER", "_LOWER"]])
    )
)
flight_emissions.head().collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO,DURATION_REFERENCE_LOWER,DURATION_REFERENCE_UPPER,INTERPOLATION_FACTOR,CO2_CCD,NOX_CCD,SOX_CCD,H2O_CCD,CO_CCD,HC_CCD,PM_NON_VOLATILE_CCD,PM_VOLATILE_CCD,PM_TOTAL_CCD
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,duration[μs],duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7c7a3f""",2026-02-19 07:15:03 AEDT,2026-02-19 08:28:20 AEDT,1h 13m 17s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.973856,9206.031679,44.69165,2.454942,3615.195082,7.933708,0.196219,0.10071,0.236895,0.337606
"""7c6db3""",2026-02-19 16:51:52 AEDT,2026-02-19 18:00:54 AEDT,1h 9m 2s,"""YMML""","""YSSY""","""B738""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 12m 25s,39m 15s,0.10201,8685.413317,42.574021,2.316111,3410.74895,7.701905,0.187078,0.097978,0.221622,0.3196
"""7cad42""",2026-02-19 19:39:26 AEDT,2026-02-19 20:46:26 AEDT,1h 7m,"""YMML""","""YSSY""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""7c47b8""",2026-02-19 09:11:53 AEDT,2026-02-19 10:29:19 AEDT,1h 17m 26s,"""YMML""","""YSSY""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""7c47ba""",2026-02-19 10:16:56 AEDT,2026-02-19 11:29:49 AEDT,1h 12m 53s,"""YMML""","""YSSY""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [39]:
# Fill in empty values with the mean of the column
cols_to_fill = [c + app for c in emissions_cols for app in ["_LTO", "_CCD"]]
flight_emissions = (
    flight_emissions
    .with_columns(
        pl.col(cols_to_fill).fill_null(pl.col(cols_to_fill).mean())
    )
)

In [40]:
flight_emissions.collect_schema().keys()

odict_keys(['ICAO24_HEX', 'DEPARTURE_TIME', 'ARRIVAL_TIME', 'FLIGHT_DURATION_ACTUAL', 'DEPARTURE_AIRPORT', 'ARRIVAL_AIRPORT', 'ICAO_OTHER', 'AIRCRAFT_ID', 'CO2_LTO', 'NOX_LTO', 'SOX_LTO', 'H2O_LTO', 'CO_LTO', 'HC_LTO', 'PM_NON_VOLATILE_LTO', 'PM_VOLATILE_LTO', 'PM_TOTAL_LTO', 'DURATION_REFERENCE_LOWER', 'DURATION_REFERENCE_UPPER', 'INTERPOLATION_FACTOR', 'CO2_CCD', 'NOX_CCD', 'SOX_CCD', 'H2O_CCD', 'CO_CCD', 'HC_CCD', 'PM_NON_VOLATILE_CCD', 'PM_VOLATILE_CCD', 'PM_TOTAL_CCD'])

## Sanity Check

What's the total emissions per day?

The CO2 in the raw data is in kg.

In [41]:
(
    flight_emissions
    .group_by(pl.col("DEPARTURE_TIME").dt.date().alias("DEPARTURE_DATE"))
    .agg([
        (pl.col(c + "_LTO").sum() + pl.col(c + "_CCD").sum()).alias(c)
        for c in emissions_cols
    ])
    .sort("DEPARTURE_DATE")
    .collect()
)

DEPARTURE_DATE,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL
date,f64,f64,f64,f64,f64,f64,f64,f64,f64
2026-02-18,21963.907567,93.03613,5.857041,8625.191656,25.756129,0.485563,0.007082,0.557624,0.564706
2026-02-19,4.5350e6,27483.572725,1209.336763,1.7809e6,4590.994197,559.233702,65.336399,133.115989,198.452388


So there's around 2.5 thousand tonnes of CO2 burnt per day.

Is that the right ballpark?

[Google Flights](https://www.google.com/travel/flights/search?tfs=CBwQAholEgoyMDI1LTExLTEyKABqDAgDEggvbS8wNnk1N3IHCAESA01FTEABSAFwAYIBCwj___________8BmAEC) says per-passenger emissions are around 77kg on average.

How many flights? Google Flights says 34 + 24 + 13 = 71 from Syd to Melb on 12/11/2025. Double that, it's 142 each way. [This source](https://www.thenewdaily.com.au/travel/2019/03/27/busiest-flight-routes-australia) says almost 150.

How many seats per flight? Google Flights assumes economy seats. How do they allocate emissions between classes? Maybe 160 passengers ([Wikipedia](https://en.wikipedia.org/wiki/Airbus_A320_family)).

In [42]:
flights_per_day = 71 * 2
passengers_per_flight = 160
emissions_per_passenger = 77

emissions_per_day = emissions_per_passenger * passengers_per_flight * flights_per_day
emissions_per_day

1749440

Ok, so we're in the right ballpark.

## Airport Data

Add data about the airports to each flight

In [43]:
airport_locations = pl.DataFrame([
    {
        "AIRPORT_ID": v[airport_id_type],
        "LATITUDE": v['location']['latitude'],
        "LONGITUDE": v['location']['longitude'],
    }
    for v in airport_info.values()
])
airport_locations

AIRPORT_ID,LATITUDE,LONGITUDE
str,f64,f64
"""YMML""",-37.673333,144.843333
"""YMAV""",-38.040556,144.470833
"""YSSY""",-33.946111,151.177222
"""YSCB""",-35.306944,149.195


In [44]:
flight_emissions = (
    flight_emissions
    .join(
        airport_locations
            .lazy()
            .rename({"AIRPORT_ID": "DEPARTURE_AIRPORT", 
                                   "LATITUDE": "LATITUDE_DEPARTURE",
                                   "LONGITUDE": "LONGITUDE_DEPARTURE"}),
        on="DEPARTURE_AIRPORT"
    )
    .join(
        airport_locations
            .lazy()
            .rename({"AIRPORT_ID": "ARRIVAL_AIRPORT",
                                   "LATITUDE": "LATITUDE_ARRIVAL", 
                                   "LONGITUDE": "LONGITUDE_ARRIVAL"}),
        on="ARRIVAL_AIRPORT"
    )
    # calculate the angle
    # 0 is east, 90 is north
    # Use arctan2 to handle all quadrants correctly
    .with_columns(
        (pl.col("LATITUDE_ARRIVAL") - pl.col("LATITUDE_DEPARTURE")).alias("LATITUDE_DELTA"),
        (pl.col("LONGITUDE_ARRIVAL") - pl.col("LONGITUDE_DEPARTURE")).alias("LONGITUDE_DELTA"),
    )
    .with_columns(
        pl.arctan2(pl.col("LATITUDE_DELTA"), pl.col("LONGITUDE_DELTA")).degrees().alias("ANGLE")
    )
)

In [45]:
flight_emissions.select("ARRIVAL_AIRPORT", "DEPARTURE_AIRPORT", "ANGLE").unique().head().collect()

ARRIVAL_AIRPORT,DEPARTURE_AIRPORT,ANGLE
str,str,f64
"""YSCB""","""YMML""",28.536865
"""YMAV""","""YSCB""",-149.944427
"""YSSY""","""YMML""",30.474986
"""YSSY""","""YMAV""",31.405276
"""YMML""","""YSSY""",-149.525014


## Passenger Count

There were 8.04 passengers (one way) per year between Sydney and Melbourne ([source](https://www.bitre.gov.au/sites/default/files/documents/domestic-aviation-activity-2024.pdf)). I assume this is both Melbourne airports.

Let's figure out the per-flight emissions. (Need to exclude flights to/from Canberra.)


In [93]:
syd_melb_passengers_yearly = 8.04e6
days_per_year = 365
syd_melb_passengers_daily = syd_melb_passengers_yearly / days_per_year

In [94]:
syd_melb_airport_ids = [v[airport_id_type] for (k, v) in airport_info.items() if k != "Canberra"]


In [95]:
melb_syd_flights_per_day = (
    flights
    .filter(pl.col("DEPARTURE_TIME").dt.date().alias("DATE") == _date)
    .filter(pl.col("DEPARTURE_AIRPORT").is_in(syd_melb_airport_ids))
    .filter(pl.col("ARRIVAL_AIRPORT").is_in(syd_melb_airport_ids))
    .select(pl.len())
    .collect()
    .item()
)
melb_syd_flights_per_day

167

In [101]:
passengers_per_flight = syd_melb_passengers_daily / melb_syd_flights_per_day

with open(passenger_numbers_results_path, 'w') as f:
    f.write(str(passengers_per_flight))

passengers_per_flight

131.90058239685015

## Mid-Flight Data

We generate a list of timestamps which we care about.

OpenSky doesn't offer mid-flight positions for historcal flights, so we'll just interpolate.

In [46]:
num_slices = 24 * 60

times_lf = pl.LazyFrame({
    "TIME": pl.datetime_range(
        start=dt.datetime.combine(_date, dt.time.min, tzinfo=sydney_tz),
        end=dt.datetime.combine(_date + dt.timedelta(days=1), dt.time.min, tzinfo=sydney_tz),
        interval=dt.timedelta(days=1) / (num_slices),
        closed='both',
        eager=True
    )
})
    
times_lf.collect()

TIME
"datetime[μs, Australia/Sydney]"
2026-02-19 00:00:00 AEDT
2026-02-19 00:01:00 AEDT
2026-02-19 00:02:00 AEDT
2026-02-19 00:03:00 AEDT
2026-02-19 00:04:00 AEDT
…
2026-02-19 23:56:00 AEDT
2026-02-19 23:57:00 AEDT
2026-02-19 23:58:00 AEDT


In [65]:
jitter_scale_end = 0.1
jitter_scale_mid = 1.5

# returns a random value [0,1] , different for each row
def random_col(scale=jitter_scale_end) -> pl.Expr:
    return (pl.int_range(pl.len()).map_batches(lambda s: pl.Series(np.random.random(len(s))), return_dtype=pl.Float64) - 0.5) * 2 * scale

animation_data = (
    flight_emissions
    .with_columns(
        random_col().alias("JITTER_LATITUDE_DEPARTURE"),
        random_col().alias("JITTER_LONGITUDE_DEPARTURE"),
        random_col().alias("JITTER_LATITUDE_ARRIVAL"),
        random_col().alias("JITTER_LONGITUDE_ARRIVAL"),
        random_col(jitter_scale_mid).alias("JITTER_LATITUDE_MID"),
        random_col(jitter_scale_mid).alias("JITTER_LONGITUDE_MID"),
    )
    .with_columns(
        (pl.col("LONGITUDE_DEPARTURE") + pl.col("JITTER_LONGITUDE_DEPARTURE")).alias("LONGITUDE_DEPARTURE"),
        (pl.col("LATITUDE_DEPARTURE") + pl.col("JITTER_LATITUDE_DEPARTURE")).alias("LATITUDE_DEPARTURE"),
        (pl.col("LONGITUDE_ARRIVAL") + pl.col("JITTER_LONGITUDE_ARRIVAL")).alias("LONGITUDE_ARRIVAL"),
        (pl.col("LATITUDE_ARRIVAL") + pl.col("JITTER_LATITUDE_ARRIVAL")).alias("LATITUDE_ARRIVAL"),
    )
    .join(times_lf, how="cross")
    .filter(pl.col("DEPARTURE_TIME") >= pl.col("TIME").min())
    .with_columns(
        (pl.col("TIME") >= pl.col("DEPARTURE_TIME")).alias("HAS_DEPARTED"),
        (pl.col("TIME") >= pl.col("ARRIVAL_TIME")).alias("HAS_ARRIVED"),
        pl.col("TIME").is_between("DEPARTURE_TIME", "ARRIVAL_TIME").alias("IN_AIR"),
        pl.col("TIME").is_between(pl.col("DEPARTURE_TIME") - taxiing_time, pl.col("DEPARTURE_TIME"), "left").alias("TAXIING_DEPARTURE"),
        pl.col("TIME").is_between(pl.col("ARRIVAL_TIME"), pl.col("ARRIVAL_TIME") + taxiing_time, "right").alias("TAXIING_ARRIVAL"),
        ((pl.col("TIME") - pl.col("DEPARTURE_TIME")) / (pl.col("ARRIVAL_TIME") - pl.col("DEPARTURE_TIME")))
        .clip(lower_bound=0, upper_bound=1)
        .alias("FLIGHT_PROGRESS")
    )
    .with_columns([
        (
            pl.col("FLIGHT_PROGRESS") * pl.col(c + "_CCD")
            + pl.col("HAS_DEPARTED") * pl.col(c + "_LTO")
        ).alias(c)
        for c in emissions_cols
    ])
    .with_columns(
        (
            pl.col("LONGITUDE_DEPARTURE") + 
            pl.col("FLIGHT_PROGRESS") * (pl.col("LONGITUDE_ARRIVAL") - pl.col("LONGITUDE_DEPARTURE")) +
            pl.col("JITTER_LATITUDE_MID") * pl.col("FLIGHT_PROGRESS") * (1 - pl.col("FLIGHT_PROGRESS"))
        ).alias("LONGITUDE"),
        (
            pl.col("LATITUDE_DEPARTURE") + 
            pl.col("FLIGHT_PROGRESS") * (pl.col("LATITUDE_ARRIVAL") - pl.col("LATITUDE_DEPARTURE")) +
            pl.col("JITTER_LONGITUDE_MID") * pl.col("FLIGHT_PROGRESS") * (1 - pl.col("FLIGHT_PROGRESS"))
        ).alias("LATITUDE"),
    )
    .sort("TIME", "DEPARTURE_TIME", "ICAO24_HEX")
)
animation_data.collect()

ICAO24_HEX,DEPARTURE_TIME,ARRIVAL_TIME,FLIGHT_DURATION_ACTUAL,DEPARTURE_AIRPORT,ARRIVAL_AIRPORT,ICAO_OTHER,AIRCRAFT_ID,CO2_LTO,NOX_LTO,SOX_LTO,H2O_LTO,CO_LTO,HC_LTO,PM_NON_VOLATILE_LTO,PM_VOLATILE_LTO,PM_TOTAL_LTO,DURATION_REFERENCE_LOWER,DURATION_REFERENCE_UPPER,INTERPOLATION_FACTOR,CO2_CCD,NOX_CCD,SOX_CCD,H2O_CCD,CO_CCD,HC_CCD,PM_NON_VOLATILE_CCD,PM_VOLATILE_CCD,PM_TOTAL_CCD,LATITUDE_DEPARTURE,LONGITUDE_DEPARTURE,LATITUDE_ARRIVAL,LONGITUDE_ARRIVAL,LATITUDE_DELTA,LONGITUDE_DELTA,ANGLE,JITTER_LATITUDE_DEPARTURE,JITTER_LONGITUDE_DEPARTURE,JITTER_LATITUDE_ARRIVAL,JITTER_LONGITUDE_ARRIVAL,JITTER_LATITUDE_MID,JITTER_LONGITUDE_MID,TIME,HAS_DEPARTED,HAS_ARRIVED,IN_AIR,TAXIING_DEPARTURE,TAXIING_ARRIVAL,FLIGHT_PROGRESS,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL,LONGITUDE,LATITUDE
str,"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]",duration[μs],str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,duration[μs],duration[μs],f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,"datetime[μs, Australia/Sydney]",bool,bool,bool,bool,bool,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""7c431c""",2026-02-19 00:32:30 AEDT,2026-02-19 01:50:27 AEDT,1h 17m 57s,"""YSSY""","""YMML""","""B463""","""A20N-A""",2081.5326,7.0691972,0.555075,817.41461,6.517649,0.092248,0.000893,0.033583,0.034476,1h 46m 11s,1h 12m 38s,0.84153,8260.545921,37.437918,2.202812,3243.903263,6.15796,0.142588,0.002459,0.225042,0.227501,-34.026782,151.116899,-37.578267,144.767361,-3.727222,-6.333889,-149.525014,-0.080671,-0.060323,0.095066,-0.075972,-0.767419,0.229982,2026-02-19 00:00:00 AEDT,false,false,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,151.116899,-34.026782
"""7c5b29""",2026-02-19 02:29:30 AEDT,2026-02-19 03:38:54 AEDT,1h 9m 24s,"""YMML""","""YSSY""","""B463""","""A20N-A""",2081.5326,7.0691972,0.555075,817.41461,6.517649,0.092248,0.000893,0.033583,0.034476,1h 12m 38s,39m,0.096135,7388.761755,34.752626,1.970336,2901.555026,5.821669,0.130812,0.002206,0.197133,0.199339,-37.727835,144.921261,-33.994838,151.210782,3.727222,6.333889,30.474986,-0.054502,0.077928,-0.048727,0.03356,-1.030697,-0.286834,2026-02-19 00:00:00 AEDT,false,false,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.921261,-37.727835
"""7c4203""",2026-02-19 05:52:31 AEDT,2026-02-19 07:09:29 AEDT,1h 16m 58s,"""YSSY""","""YMML""","""B737""","""A21N-A""",2359.83888,9.4724002,0.62929,926.70503,7.9447,0.09561,0.034621,0.03837,0.072992,1h 45m 34s,1h 12m 25s,0.862745,9674.588374,46.619985,2.579891,3799.196513,8.097909,0.203814,0.103495,0.250404,0.353899,-33.959415,151.255379,-37.740364,144.942573,-3.727222,-6.333889,-149.525014,-0.013304,0.078157,-0.067031,0.09924,1.363769,0.019721,2026-02-19 00:00:00 AEDT,false,false,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,151.255379,-33.959415
"""7c7701""",2026-02-19 05:56:40 AEDT,2026-02-19 07:12:11 AEDT,1h 15m 31s,"""YSSY""","""YMML""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 16m 56s,40m 42s,0.039098,44524.976589,333.522082,11.873327,17484.886994,16.549564,8.415624,0.345408,1.785794,2.131202,-33.97383,151.125002,-37.637207,144.917907,-3.727222,-6.333889,-149.525014,-0.027719,-0.05222,0.036126,0.074574,-0.11327,0.598596,2026-02-19 00:00:00 AEDT,false,false,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,151.125002,-33.97383
"""7c6c90""",2026-02-19 06:01:32 AEDT,2026-02-19 07:06:54 AEDT,1h 5m 22s,"""YMAV""","""YSSY""","""A320""","""A124-A""",10769.22,67.73848,2.871792,4229.0554,20.070542,3.73421,0.10514,0.244812,0.3499524,1h 16m 56s,40m 42s,0.319227,38583.881369,295.249919,10.289035,15151.828625,14.798195,7.340495,0.311323,1.51998,1.831303,-38.078062,144.375689,-33.964511,151.207744,4.094445,6.706389,31.405276,-0.037506,-0.095144,-0.0184,0.030522,-0.361084,1.444333,2026-02-19 00:00:00 AEDT,false,false,false,false,false,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,144.375689,-38.078062
…,…,…,…

`animation_data` has one row per (flight, time). The emissions values are cumulative, within each flight. So for total emissions, group by time, sum across flights.

In [66]:
os.makedirs(results_dir, exist_ok=True)

In [67]:
(
    animation_data
    .select(
        pl.col("TIME").min().alias("START_TIME"),
        pl.col("TIME").max().alias("END_TIME"),
    )
    .collect()
)

START_TIME,END_TIME
"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]"
2026-02-19 00:00:00 AEDT,2026-02-20 00:00:00 AEDT


In [104]:
(
    animation_data
    .group_by("TIME")
    .agg([
        pl.col(c).sum()
        for c in emissions_cols
    ] + [
        pl.struct("ICAO24_HEX", "DEPARTURE_TIME", "ARRIVAL_TIME").filter("HAS_DEPARTED").n_unique().alias("NUM_FLIGHTS"),
        (pl.col("HAS_ARRIVED").sum() * passengers_per_flight).round().alias("PASSENGERS_ARRIVED").cast(int)
    ])
    # # reset counter to zero (some flights prior to the start time were counted)
    # .sort("TIME")
    # .with_columns(
    #     (pl.exclude("TIME") - pl.exclude("TIME").first())
    # )
    .sort("TIME")
    .sink_parquet(emissions_results_path)
)
animation_emissions_data = pl.read_parquet(emissions_results_path)
animation_emissions_data

TIME,CO2,NOX,SOX,H2O,CO,HC,PM_NON_VOLATILE,PM_VOLATILE,PM_TOTAL,NUM_FLIGHTS,PASSENGERS_ARRIVED
"datetime[μs, Australia/Sydney]",f64,f64,f64,f64,f64,f64,f64,f64,f64,u32,i64
2026-02-19 00:00:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2026-02-19 00:01:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2026-02-19 00:02:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2026-02-19 00:03:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
2026-02-19 00:04:00 AEDT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0
…,…,…,…,…,…,…,…,…,…,…,…
2026-02-19 23:56:00 AEDT,4.5350e6,27483.572725,1209.336763,1.7809e6,4590.994197,559.233702,65.336399,133.115989,198.452388,239,31524
2026-02-19 23:57:00 AEDT,4.5350e6,27483.572725,1209.336763,1.7809e6,4590.994197,559.233702,65.336399,133.115989,198.452388,239,31524
2026-02-19 23:58:00 AEDT,4.5350e6,27483.572725,1209.336763,1.7809e6,4590.994197,559.233702,65.336399,133.115989,198.452388,239,31524


In [71]:
(
    animation_data
    .sort("ICAO24_HEX", "DEPARTURE_TIME", "TIME")
    .select("TIME", "LATITUDE", "LONGITUDE", "ANGLE", "IN_AIR", "TAXIING_ARRIVAL", "TAXIING_DEPARTURE")
    .sink_parquet(planes_results_path)
)
animation_plane_data = pl.read_parquet(planes_results_path)
animation_plane_data

TIME,LATITUDE,LONGITUDE,ANGLE,IN_AIR,TAXIING_ARRIVAL,TAXIING_DEPARTURE
"datetime[μs, Australia/Sydney]",f64,f64,f64,bool,bool,bool
2026-02-19 00:00:00 AEDT,-37.686295,144.807308,28.536865,false,false,false
2026-02-19 00:01:00 AEDT,-37.686295,144.807308,28.536865,false,false,false
2026-02-19 00:02:00 AEDT,-37.686295,144.807308,28.536865,false,false,false
2026-02-19 00:03:00 AEDT,-37.686295,144.807308,28.536865,false,false,false
2026-02-19 00:04:00 AEDT,-37.686295,144.807308,28.536865,false,false,false
…,…,…,…,…,…,…
2026-02-19 23:56:00 AEDT,-35.24283,149.16891,28.536865,false,false,false
2026-02-19 23:57:00 AEDT,-35.24283,149.16891,28.536865,false,false,false
2026-02-19 23:58:00 AEDT,-35.24283,149.16891,28.536865,false,false,false


In [72]:
(
    pl.scan_parquet(emissions_results_path)
    .select(
        pl.col("TIME").min().alias("START_TIME"),
        pl.col("TIME").max().alias("END_TIME"),
    )
    .collect()
)

START_TIME,END_TIME
"datetime[μs, Australia/Sydney]","datetime[μs, Australia/Sydney]"
2026-02-19 00:00:00 AEDT,2026-02-20 00:00:00 AEDT
